# Dataset Analysis

This notebook uses the preprocessing logic to analyze the currently available data in the `data` folder.

In [ ]:
# Install dependencies in the current Jupyter kernel environment
%pip install -r ../requirements.txt

In [ ]:
import sys
import os

# Add the project root to path so we can import model.preprocessing
sys.path.append(os.path.abspath('..'))

from model.preprocessing import analyze_data

In [ ]:
# Run the analysis on the 'data' directory located in the project root
stats = analyze_data('../data')

print("=== Dataset Analysis ===")
print(f"Total number of entries in our dataset: {stats['total_samples']}\n")

print("=== Class Distribution ===")
for category, count in stats['class_counts'].items():
    print(f"{category}: {count}")

print("\n=== Missing Fields ===")
for field, count in stats['missing_fields'].items():
    print(f"{field}: {count}")


# Class Imbalance Analysis

Yes. You can split your dataset 80/20 without manually balancing it first, but I strongly recommend doing a stratified 80/20 split so each class keeps roughly the same proportion in both sets.

And I know it's not severely imbalanced because of the class counts:

- Accommodation: 423
- Culinary: 346
- Adventure: 298
- Urban: 285
- Coastal: 282
- Cultural: 274
- OUT_OF_SCOPE: 226
- Theme Parks: 155

The largest class has 423 examples, while the smallest has 155.

So:
423 ÷ 155 ≈ 2.73

That means your largest class is about 2.7× the size of your smallest class. That's an imbalance, but not an extreme imbalance.

For your case:
1,383 total → stratified 80/20 split → ~1,106 training + ~277 validation/test

You don't need to artificially duplicate or remove data just to make every class equal. Start with the stratified split and evaluate the F1-score for each class.

There isn't one universal cutoff, but a common practical way to judge class imbalance is by looking at the largest-to-smallest class ratio.

For example:
- Largest : Smallest 1:1 – 2:1 🟢 Very balanced
- 2:1 – 5:1 🟡 Mild/moderate imbalance
- 5:1 – 10:1 🟠 Significant imbalance
- >10:1 🔴 Severe imbalance

These aren't strict mathematical rules—the impact on your model matters more than the ratio alone.


# Dataset Consolidation

Aggregating all the fragmented batch JSON files from the `data/` folder into a single, consolidated dataset. This is highly recommended because:
1. **Single Source of Truth**: It removes the need to parse dozens of files repeatedly.
2. **Easier Splitting**: A single array of data is much easier to load into `scikit-learn` for the stratified 80/20 split.
3. **Data Integrity**: We can quickly inspect one file to see all 1,383 entries.

*(Note: We are saving the consolidated text data here. We do not save the tokenized PyTorch tensors to JSON, as JSON is highly inefficient for storing large numerical arrays. Tokenization will happen on-the-fly during training, or could be cached as `.pt` files if needed).*

In [ ]:
import json
import os
import sys
sys.path.append(os.path.abspath('..'))
from model.preprocessing import load_data_from_json


# Create dataset directory if it doesn't exist
dataset_dir = '../dataset'
os.makedirs(dataset_dir, exist_ok=True)

# Load all data using the existing preprocessing function
# This function already reads all JSONs in the folder and aggregates them
data_list = load_data_from_json('../data')

# Save to a single consolidated JSON file
output_path = os.path.join(dataset_dir, 'consolidated_tourism_data.json')

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(data_list, f, indent=4, ensure_ascii=False)

print(f"Successfully consolidated {len(data_list)} entries into {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.2f} KB")
